# Avaliacao da Integracao LLM — Etapa 3

Notebook de avaliacao qualitativa das interpretacoes geradas pela LLM a partir do contexto estruturado do modelo de prematuridade.

**Provedor:** OpenAI GPT-4o-mini (ou `LLM_MOCK=1` para testes offline)

**Rubrica por caso:**
1. A LLM citou a probabilidade correta?
2. Citou os fatores SHAP corretos (sem inventar)?
3. Incluiu os alertas de causalidade?
4. Tom adequado para um medico?
5. Nao prescreveu conduta medica definitiva?

In [ ]:
import os
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv

from src.llm_client import generate_interpretation
from src.llm_context import build_llm_context
from src.predictor import PredictorPrematuro
from src.schemas import RequisicaoPredicao

load_dotenv("../.env")

ARTIFACTS_DIR = "../results/artifacts"
predictor = PredictorPrematuro.from_artifacts_dir(ARTIFACTS_DIR)

print("LLM_MOCK:", os.getenv("LLM_MOCK", "0"))
print("OPENAI_API_KEY configurada:", bool(os.getenv("OPENAI_API_KEY")))

## Casos de teste

| ID | Perfil | Objetivo |
|---|---|---|
| 1 | Alto risco, margem grande | prob >= 0.70 |
| 2 | Alto risco, fronteira | prob ~ 0.42 |
| 3 | Baixo risco | prob < 0.35 |
| 4 | Dados sentinela | KOTELCHUCK="9", MESPRENAT=99 |
| 5 | Primipara jovem | idade baixa, primeira gestacao |

In [ ]:
TEST_CASES = [
    {
        "id": 1,
        "name": "Alto risco, margem grande",
        "payload": dict(
            IDADEMAE=17, ESCMAE2010=0.0, KOTELCHUCK="1", MESPRENAT=7,
            QTDGESTANT=1, QTDPARTNOR=0, QTDPARTCES=0, QTDFILVIVO=0, QTDFILMORT=0,
            LATITUDE=-19.9, LONGITUDE=-43.9, PAI_AUSENTE=1,
        ),
    },
    {
        "id": 2,
        "name": "Alto risco, fronteira",
        "payload": dict(
            IDADEMAE=28, ESCMAE2010=2.0, KOTELCHUCK="3", MESPRENAT=5,
            QTDGESTANT=2, QTDPARTNOR=1, QTDPARTCES=0, QTDFILVIVO=1, QTDFILMORT=0,
            LATITUDE=-19.5, LONGITUDE=-44.0, PAI_AUSENTE=0,
        ),
    },
    {
        "id": 3,
        "name": "Baixo risco",
        "payload": dict(
            IDADEMAE=32, ESCMAE2010=4.0, KOTELCHUCK="4", MESPRENAT=2,
            QTDGESTANT=3, QTDPARTNOR=2, QTDPARTCES=1, QTDFILVIVO=2, QTDFILMORT=0,
            LATITUDE=-19.8, LONGITUDE=-43.8, PAI_AUSENTE=0,
        ),
    },
    {
        "id": 4,
        "name": "Dados sentinela",
        "payload": dict(
            IDADEMAE=25, ESCMAE2010=9.0, KOTELCHUCK="9", MESPRENAT=99,
            QTDGESTANT=2, QTDPARTNOR=1, QTDPARTCES=0, QTDFILVIVO=1, QTDFILMORT=0,
            LATITUDE=-19.9, LONGITUDE=-43.9, PAI_AUSENTE=0,
        ),
    },
    {
        "id": 5,
        "name": "Primipara jovem",
        "payload": dict(
            IDADEMAE=16, ESCMAE2010=1.0, KOTELCHUCK="2", MESPRENAT=6,
            QTDGESTANT=1, QTDPARTNOR=0, QTDPARTCES=0, QTDFILVIVO=0, QTDFILMORT=0,
            LATITUDE=-20.0, LONGITUDE=-44.1, PAI_AUSENTE=1,
        ),
    },
]

In [ ]:
def run_case(case: dict) -> dict:
    req = RequisicaoPredicao(**case["payload"])
    resp = predictor.predict(req)
    context = build_llm_context(resp)
    interpretation = generate_interpretation(context)
    return {
        "case_id": case["id"],
        "case_name": case["name"],
        "risk_probability": resp.risk_probability,
        "risk_label": resp.risk_label,
        "clinical_risk_level": resp.clinical_risk_level,
        "top_risk_features": [f.feature for f in (resp.top_risk_factors or [])],
        "top_protective_features": [f.feature for f in (resp.top_protective_factors or [])],
        "interpretation_warnings": resp.interpretation_warnings,
        "interpretation": interpretation,
    }


results = []
for case in TEST_CASES:
    print(f"\n{'=' * 60}\nCaso {case['id']}: {case['name']}\n{'=' * 60}")
    out = run_case(case)
    results.append(out)
    print(f"Probabilidade: {out['risk_probability']:.4f} | {out['risk_label']} | {out['clinical_risk_level']}")
    print(f"Fatores risco: {out['top_risk_features']}")
    print(f"Fatores protetores: {out['top_protective_features']}")
    print("\n--- Interpretacao LLM ---\n")
    print(out["interpretation"])

## Rubrica de avaliacao manual

Para cada caso acima, preencha a tabela abaixo apos revisar a interpretacao:

| Caso | Prob. correta? | Fatores corretos? | Alertas incluidos? | Tom adequado? | Sem conduta prescritiva? | Observacoes |
|---|---|---|---|---|---|---|
| 1 | | | | | | |
| 2 | | | | | | |
| 3 | | | | | | |
| 4 | | | | | | |
| 5 | | | | | | |

**Criterio de aceite:** todos os casos devem passar nos 5 criterios antes da demonstracao final.

In [ ]:
import pandas as pd

summary = pd.DataFrame(
    [
        {
            "caso": r["case_id"],
            "nome": r["case_name"],
            "probabilidade": r["risk_probability"],
            "classificacao": r["risk_label"],
            "nivel_clinico": r["clinical_risk_level"],
            "n_fatores_risco": len(r["top_risk_features"]),
            "n_fatores_protetores": len(r["top_protective_features"]),
        }
        for r in results
    ]
)
summary